# Uber Weekly Ridership Prediction

This notebook explores the `taxi.csv` dataset and trains a regression model to predict the **number of weekly riders** from four socio-economic features:

| Feature | Meaning |
|---|---|
| `Priceperweek` | Weekly ticket price ($) |
| `Population` | City population |
| `Monthlyincome` | Average monthly income ($) |
| `Averageparkingpermonth` | Average monthly parking cost ($) |

**Target:** `Numberofweeklyriders`

> The production training logic lives in `src/train.py`. This notebook mirrors that workflow for exploration and explanation.

## 1. Setup & data loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_csv("taxi.csv")
print(df.shape)
df.head()

In [ ]:
df.describe()

## 2. Exploratory data analysis

The dataset is small (~27 rows), so we focus on relationships rather than distributions.

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlation")
plt.tight_layout()
plt.show()

In [ ]:
features = ["Priceperweek", "Population", "Monthlyincome", "Averageparkingpermonth"]
target = "Numberofweeklyriders"

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.ravel(), features):
    sns.regplot(data=df, x=col, y=target, ax=ax, scatter_kws={"alpha": 0.7})
    ax.set_title(f"{col} vs {target}")
plt.tight_layout()
plt.show()

**Observation:** `Priceperweek` and `Monthlyincome` are strongly *negatively* correlated with ridership (higher prices/incomes coincide with fewer transit riders), while `Population` is positively correlated.

## 3. Model comparison

We compare four pipelines (each standard-scales its inputs) with repeated K-fold cross-validation, exactly as `src/train.py` does.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedKFold, cross_validate

X, y = df[features], df[target]

def pipe(model):
    return Pipeline([("scaler", StandardScaler()), ("model", model)])

candidates = {
    "LinearRegression": pipe(LinearRegression()),
    "Ridge": pipe(Ridge(alpha=1.0, random_state=42)),
    "RandomForest": pipe(RandomForestRegressor(n_estimators=300, random_state=42)),
    "GradientBoosting": pipe(GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42)),
}

cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)
rows = []
for name, p in candidates.items():
    s = cross_validate(p, X, y, cv=cv, scoring=["r2", "neg_root_mean_squared_error"])
    rows.append({
        "model": name,
        "r2_mean": s["test_r2"].mean(),
        "rmse_mean": -s["test_neg_root_mean_squared_error"].mean(),
    })

leaderboard = pd.DataFrame(rows).sort_values("r2_mean", ascending=False).reset_index(drop=True)
leaderboard

## 4. Fit the winner and persist it

The best model is refit on the full dataset and saved to `model.pkl`. In production this is done by `python -m src.train`, which also writes `metrics.json`.

In [ ]:
import joblib

best_name = leaderboard.iloc[0]["model"]
best = candidates[best_name].fit(X, y)
joblib.dump(best, "model.pkl")
print(f"Saved {best_name} -> model.pkl")

In [ ]:
# Sanity-check a single prediction
sample = pd.DataFrame([[25, 1800000, 12000, 90]], columns=features)
print("Predicted weekly riders:", round(best.predict(sample)[0]))

## 5. Next steps

- Collect more data — 27 rows limits model complexity and confidence.
- Add feature engineering (e.g. price-to-income ratio).
- Serve predictions via the Flask app: `python app.py` then open http://localhost:5000.